In [1]:
## Import modules
import os, sys
import numpy as np
import geopandas as gpd
import cftime
import gc
import shapely
import json
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
import glob
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as pc
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox
import ipywidgets as widgets
import psutil
import time 
import numba
import platform

cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))
sys.path.insert(0, cmct_dir)

from cmct.calving import *
# Detect and configure GPU
gpu_config = detect_and_configure_gpu()

# Store GPU config as environment variable for calving modules
os.environ['CMCT_GPU_PLATFORM'] = gpu_config['platform']
os.environ['CMCT_CUDA_AVAILABLE'] = str(gpu_config['cuda_available'])
os.environ['CMCT_METAL_AVAILABLE'] = str(gpu_config['metal_available'])

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir

# Initialising Logger
import logging

logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Import utilities for this comparison
from cmct.time_utils import *
from cmct.calving_modules.interpolation import *
from cmct.calving_modules.residual_calculation import *

# Force initial garbage collection
gc.collect()

# Store GPU config globally for use in other cells
globals()['GPU_CONFIG'] = gpu_config

In [9]:
# Reload modules to pick up any changes to imports
import importlib
import sys

# Clear module from cache to force reload
if 'cmct.calving' in sys.modules:
    del sys.modules['cmct.calving']
if 'cmct.calving_modules.residual_calculation' in sys.modules:
    del sys.modules['cmct.calving_modules.residual_calculation']

import cmct.calving
import cmct.calving_modules.residual_calculation
from cmct.calving_modules.plotting_utils import *
from cmct.calving import calculate_basin_statistics, format_basin_stats
from cmct.calving import calculate_basin_statistics

importlib.reload(cmct.calving)
importlib.reload(cmct.calving_modules.residual_calculation)

# Re-import to ensure functions are available
from cmct.calving import *

# Memory monitoring utility
def get_memory_usage():
    """Get current memory usage in MB"""
    import psutil
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def log_memory_usage(stage=""):
    """Log current memory usage"""
    try:
        mem_usage = get_memory_usage()
        logging.info(f"Memory usage {stage}: {mem_usage:.1f} MB")
    except ImportError:
        logging.warning("psutil not available for memory monitoring")
    except Exception as e:
        logging.error(f"Error checking memory: {e}")

# Check initial memory usage
log_memory_usage("at start")

# CONFIG

In [10]:
# Observation Dataset
# Ice sheet
loc = "GIS"  # 'GIS' or 'AIS'

# Set the observation data dir path
obs_filename = cmct_dir + "/data/calving/observed_icemask_ismip_annual.nc"

# To use aggregation functions for basin
basin_aggregation = True  # IMPORTANT

basin_filename = (
    cmct_dir + "/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp"
)

# Set the Model Data dir path
# model_filename = cmct_dir + "/test/calving/ensemble/sftgif_B001_hist.nc"
model_files_dir = cmct_dir + "/test/calving/ensemble/*.nc"

# Set time range for comparison
start_year = 2007
end_year = 2015

# List of basins (ex ["NW", "NE"]) to compare if all -> "all", if none -> False
# If you do not know which basins are in the model, you can put "auto"
# NOTE: Align this list with the basins in the model.
basin_list = ["NW"]

# Output filetype and filename
filetype = "netcdf"  # netcdf or json or None
filename = "calving_comparison"

# Optional Configurations
interpolation_method = "slinear"  # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = "mean"  # 'mean', 'RMS',

colors = {
    "CW": "blue",
    "NE": "red",
    "SE": "green",
    "SW": "orange",
    "NO": "purple",
    "NW": "brown",
}

plt.figure(figsize=(12, 8))

MAX_CPU = 80
MAX_MEM = 80




<Figure size 1200x800 with 0 Axes>

In [11]:
# Check if observation file exist
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")


if basin_aggregation and not os.path.exists(basin_filename):
    raise FileNotFoundError(f"Basin shapefile not found: {basin_filename}")
    # Load basin shapes

print(basin_filename)
basins, basin_list = load_basins(basin_filename, basin_list)

print(obs_filename)
gsfc = load_gsfc_calving(obs_filename, basins)
gsfc.ds["time"] = standardising_time_var(gsfc.time)

gsfc_stats = calculate_gsfc_statistics(gsfc, basins)



/Users/aditya_pachpande/Documents/GitHub/CmCt/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp
/Users/aditya_pachpande/Documents/GitHub/CmCt/data/calving/observed_icemask_ismip_annual.nc


In [12]:
model_files = model_files_dir
model_files = glob.glob(model_files)
# Sort the files for consistent ordering
model_files.sort()
# Generate model names from filenames (extract basename without extension)
model_names = [os.path.splitext(os.path.basename(f))[0] for f in model_files]

# Validate that we found model files
if not model_files:
    raise FileNotFoundError(
        f"No model files found matching pattern: {model_files}"
    )

logging.info(f"Found {len(model_files)} model files:")
for file in model_names:
    logging.info(os.path.basename(file))
    

In [13]:
def compute_basin_mask_for_ensemble(
    gsfc, first_model_file, basin_polygons_dict, start_year
):
    """
    Compute basin mask once using the first model file.

    Parameters
    ----------
    gsfc : GSFCcalving
        GSFC calving data object
    first_model_file : str
        Path to the first model file
    basin_polygons_dict : dict
        Dictionary of basin polygons
    start_year : int
        Year to use for mask computation

    Returns
    -------
    tuple
        (basin_mask, basin_names, x_coords, y_coords)
    """
    logging.info("Computing basin mask using first ensemble member...")
    # Load first model
    first_model = load_model_calving(first_model_file)

    interpolater = Interpolater(first_model, gsfc)
    first_model.ds = interpolater.interpolate()

    if first_model is None or first_model.ds is None:
        raise ValueError(f"Failed to load first model from {first_model_file}")

    first_model.ds["time"] = standardising_time_var(first_model.time)

    # Compute basin mask using the residual_calc module
    basin_mask, basin_names, x_coords, y_coords = compute_basin_mask_once(
        gsfc, first_model, basin_polygons_dict, year=start_year
    )

    logging.info(f"Basin mask computed with shape: {basin_mask.shape}")
    logging.info(f"Number of basins: {len(basin_names)}")
    logging.info(f"Basins: {', '.join(basin_names)}")

    del first_model
    gc.collect()

    return basin_mask, basin_names, x_coords, y_coords

basin_mask, basin_names, x_coords, y_coords = compute_basin_mask_for_ensemble(
    gsfc, model_files[0], basins, start_year
)

2025-07-31 10:13:32,520 - WARNING - Checking coordinate alignment for shape trimming...


In [14]:
def process_single_model_with_mask(
    file, gsfc, basin_mask, basin_names, x_coords, y_coords, start_year, end_year
):
    """
    Process a single model file using pre-computed basin mask.

    Parameters
    ----------
    file : str
        Path to the model file
    gsfc : GSFCcalving
        GSFC calving data object
    basin_mask : np.ndarray
        Pre-computed basin mask
    basin_names : list
        List of basin names
    x_coords : np.ndarray
        X coordinates
    y_coords : np.ndarray
        Y coordinates
    start_year : int
        Start year for analysis
    end_year : int
        End year for analysis

    Returns
    -------
    dict
        Basin statistics for the model
    """
    logging.info(f"Processing: {file}")

    try:
        model_res = load_model_calving(file)

        # Validate model data
        if model_res is None or model_res.ds is None:
            raise ValueError(f"Failed to load model data from {file}")

        model_res.ds["time"] = standardising_time_var(model_res.time)

        # Handle Time Range with validation
        try:
            checking_calving_daterange(
                gsfc.time.values, model_res.time.values, start_year, end_year
            )
        except Exception as e:
            logging.warning(f"Warning: Time range validation failed: {e}")

        # Interpolation with error handling
        try:
            interpolater = Interpolater(model_res, gsfc)
            model_res.ds = interpolater.interpolate()
            logging.info(f"Resampled data shape: {model_res.ds.dims}")
        except Exception as e:
            logging.warning(f"Warning: Interpolation failed: {e}")
            # Continue with original resolution if interpolation fails

        years = np.arange(start_year, end_year + 1)

        # Create residuals dataset using pre-computed basin mask
        try:
            residuals_dataset = create_calving_dataset_with_precomputed_mask(
                gsfc, model_res, years, basin_mask, basin_names, x_coords, y_coords
            )

            # Validate dataset
            if residuals_dataset is None:
                raise ValueError("Failed to create residuals dataset")

        except Exception as e:
            logging.error(f"Error creating residuals dataset: {e}")
            # Return empty stats if dataset creation fails
            return {
                year: {"NW": {"count": 0, "mean": np.nan, "std": np.nan}}
                for year in years
            }

        # Load residuals with error handling
        try:
            residuals = load_residuals(residuals_dataset)
            if residuals is None:
                raise ValueError("Failed to load residuals")
        except Exception as e:
            logging.error(f"Error loading residuals: {e}")
            return {
                year: {"NW": {"count": 0, "mean": np.nan, "std": np.nan}}
                for year in years
            }

        # Calculate basin statistics with error handling
        try:
            basin_stats = calculate_basin_statistics(residuals)
            logging.info(format_basin_stats(basin_stats))
        except Exception as e:
            logging.error(f"Error calculating basin statistics: {e}")
            basin_stats = {
                year: {"NW": {"count": 0, "mean": np.nan, "std": np.nan}}
                for year in years
            }

        # Memory cleanup
        del model_res, interpolater, residuals_dataset, residuals
        gc.collect()

        return basin_stats

    except Exception as e:
        logging.error(f"Error processing model file {file}: {e}")
        # Return fallback statistics
        years = np.arange(start_year, end_year + 1)
        return {
            year: {"NW": {"count": 0, "mean": np.nan, "std": np.nan}} for year in years
        }


In [15]:
basin_stats_array = []
failed_files = []

for i, file in enumerate(model_files):
    try:
        if not os.path.exists(file):
            raise FileNotFoundError(f"Model file not found: {file}")

        print(f"\n[{i + 1}/{len(model_files)}] Processing model file: {os.path.basename(file)}")

        # Process with timeout and memory monitoring
        basin_stats = process_single_model_with_mask(
            file,
            gsfc,
            basin_mask,
            basin_names,
            x_coords,
            y_coords,
            start_year,
            end_year,
        )

        # Validate the returned statistics
        if basin_stats and any(
            year in basin_stats for year in range(start_year, end_year + 1)
        ):
            basin_stats_array.append(basin_stats)
            logging.info(f"Successfully processed {os.path.basename(file)}")
        else:
            logging.warning(f"Invalid statistics returned for {os.path.basename(file)}")
            failed_files.append(file)

        del basin_stats
        gc.collect()

        cpu = psutil.cpu_percent()
        mem = psutil.virtual_memory().percent

        if cpu > MAX_CPU or mem > MAX_MEM:
            logging.warning(f"High resource usage (CPU: {cpu:.1f}%, MEM: {mem:.1f}%), waiting...")
            time.sleep(5)

    except Exception as e:
        logging.error(f"Error processing model file {os.path.basename(file)}: {e}")
        failed_files.append(file)

        # Force cleanup on error
        gc.collect()
        continue



[1/5] Processing model file: sftgif_B001_hist.nc


2025-07-31 10:13:57,861 - WARNING - Checking coordinate alignment for shape trimming...


The selected dates 2007 to 2015 are within the overlapping data range.

[2/5] Processing model file: sftgif_B002_hist.nc

[2/5] Processing model file: sftgif_B002_hist.nc


2025-07-31 10:14:03,232 - WARNING - Checking coordinate alignment for shape trimming...


The selected dates 2007 to 2015 are within the overlapping data range.

[3/5] Processing model file: sftgif_B003_hist.nc

[3/5] Processing model file: sftgif_B003_hist.nc


...


[4/5] Processing model file: sftgif_B004_hist.nc


2025-07-31 10:14:17,661 - WARNING - Checking coordinate alignment for shape trimming...


The selected dates 2007 to 2015 are within the overlapping data range.

[5/5] Processing model file: sftgif_B005_hist.nc

[5/5] Processing model file: sftgif_B005_hist.nc


2025-07-31 10:14:22,311 - WARNING - Checking coordinate alignment for shape trimming...


The selected dates 2007 to 2015 are within the overlapping data range.


# Ensemble Plotting and Analysis

In [16]:
# Reload the plotting module to ensure latest changes are available
import importlib
import cmct.calving_modules.plotting_utils
importlib.reload(cmct.calving_modules.plotting_utils)

# Import ensemble plotting utilities
from cmct.calving_modules.plotting_utils import (
    create_ensemble_time_series_plot,
    create_interactive_ensemble_plot,
    create_ensemble_statistics_summary,
)

## PLOTTING CONFIGURATIONS

In [17]:
# Create interactive ensemble plot without GSFC line
logging.info(f"Models: {model_names}")
logging.info(f"Basins: {basin_list}")

# Convert NumPy array to Python list if needed
if isinstance(basin_stats_array, np.ndarray):
    basin_stats_list = basin_stats_array.tolist()
    logging.info("Converted NumPy array to Python list")
else:
    basin_stats_list = basin_stats_array
    logging.info("Using existing list")

# Create the interactive plot without GSFC
ensemble_plot_widget = create_interactive_ensemble_plot(
    basin_stats_list, 
    model_names, 
    basin_list=basin_list, 
    gsfc_stats=None  # Remove GSFC line
)

# Display the widget
ensemble_plot_widget

In [18]:
# Reload the plotting module to get the new advanced plotting function
import importlib
import cmct.calving_modules.plotting_utils
importlib.reload(cmct.calving_modules.plotting_utils)

# Import the new advanced plotting function
from cmct.calving_modules.plotting_utils import create_advanced_ensemble_comparison_plot

In [19]:
# Create the advanced interactive comparison widget
advanced_comparison_widget = create_advanced_ensemble_comparison_plot(
    basin_stats_array,
    model_names,
    basin_list=basin_list,
    start_year=start_year,
    end_year=end_year
)

# Display the widget
advanced_comparison_widget

The end for now